# IK MLP Training

In [2]:

!git clone https://github.com/max15189/InverseKinematics.git
import sys
sys.path.insert(0, '/content/InverseKinematics')
from google.colab import drive
drive.mount('/content/drive')

Cloning into 'InverseKinematics'...
remote: Enumerating objects: 68, done.
remote: Counting objects: 100% (68/68), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 68 (delta 26), reused 57 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (68/68), 239.70 KiB | 6.15 MiB/s, done.
Resolving deltas: 100% (26/26), done.
Mounted at /content/drive


## Config

In [5]:
SAVE_DATA = "/content/drive/MyDrive/inverse_kinematics/pos_rot_qinit"
SAVE_MODEL = f"{SAVE_DATA}/mlp_ik.pt"

BATCH_SIZE = 512
EPOCHS     = 200
LR         = 1e-3
PATIENCE   = 30

## Imports

In [3]:
import sys
sys.path.insert(0, r"c:\Users\max\InverseKinematics")

import torch
from torch.utils.data import DataLoader

from ik.data.dataset import IKDataset
from ik.model.mlp import MLP
from ik.model.training import run_training, evaluate_and_return_loss

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEVICE}")

device: cpu


## Load Data

In [6]:
train_ds = IKDataset("train", SAVE_DATA)
val_ds   = IKDataset("val",   SAVE_DATA,
                     MinMax_X=train_ds.MinMax_X,
                     MinMax_Y=train_ds.MinMax_Y)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"train: {len(train_ds):,} samples")
print(f"val:   {len(val_ds):,} samples")
print(f"input dim: {train_ds.X.shape[1]}  (R6[6] + P[3] + q_init[6])")
print(f"output dim: {train_ds.y.shape[1]}  (q_target[6])")

train: 240,000 samples
val:   30,000 samples
input dim: 15  (R6[6] + P[3] + q_init[6])
output dim: 6  (q_target[6])


## Model

In [7]:
model = MLP(input_dim=15, hidden_dim=256, output_dim=6, n_layers=4).to(DEVICE)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")
print(model)

parameters: 203,014
MLP(
  (net): Sequential(
    (0): Linear(in_features=15, out_features=256, bias=True)
    (1): LeakyReLU(negative_slope=0.01)
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): LeakyReLU(negative_slope=0.01)
    (4): Linear(in_features=256, out_features=256, bias=True)
    (5): LeakyReLU(negative_slope=0.01)
    (6): Linear(in_features=256, out_features=256, bias=True)
    (7): LeakyReLU(negative_slope=0.01)
    (8): Linear(in_features=256, out_features=6, bias=True)
  )
)


## Train

In [ ]:
model = run_training(
    model,
    train_loader,
    val_loader,
    MinMax_Y=train_ds.MinMax_Y,
    device=DEVICE,
    save_path=SAVE_MODEL,
    epochs=EPOCHS,
    lr=LR,
    patience=PATIENCE,
)

## Evaluate on Val Set

In [ ]:
mse_losses, pos_losses, rot_losses = evaluate_and_return_loss(
    model, val_loader, DEVICE, MinMax_Y=train_ds.MinMax_Y
)

In [9]:
model.load_state_dict(torch.load(SAVE_MODEL))

<All keys matched successfully>

In [11]:
train_ds.MinMax_Y

[array([-3.1415927, -1.7627826, -1.7627826, -3.1415927, -1.8675023,
        -3.1415927], dtype=float32),
 array([3.1415927, 1.7627826, 1.6057029, 3.1415927, 2.268928 , 3.1415927],
       dtype=float32)]

In [19]:
from ik.model.training import _denorm_q,_to_tensors
test_ds=IKDataset("test",SAVE_DATA,train_ds.MinMax_X,train_ds.MinMax_Y)
Y_min,Y_max=_to_tensors(test_ds.MinMax_Y,DEVICE)
q_predict=model(torch.tensor(test_ds.X))
q_predict=_denorm_q(q_predict,Y_min,Y_max)
q_predict=q_predict.cpu().detach().numpy()

IKinSpace from moderin roboitcs library with iteration in the return

In [20]:
!pip install modern-robotics
import numpy as np
import modern_robotics as mr
def IKinSpace_with_iters(Slist, M, T, thetalist0, eomg, ev):
    thetalist = np.array(thetalist0).copy()
    i = 0
    maxiterations = 50
    Tsb = mr.FKinSpace(M,Slist, thetalist)
    Vs = np.dot(mr.Adjoint(Tsb),mr.se3ToVec(mr.MatrixLog6(np.dot(mr.TransInv(Tsb), T))))
    err = np.linalg.norm([Vs[0], Vs[1], Vs[2]]) > eomg or np.linalg.norm([Vs[3], Vs[4], Vs[5]]) >  ev
    while err and i < maxiterations:
        thetalist = thetalist + np.dot(np.linalg.pinv(mr.JacobianSpace(Slist,thetalist)), Vs)
        i = i + 1
        Tsb = mr.FKinSpace(M, Slist, thetalist)
        Vs = np.dot(mr.Adjoint(Tsb), \
                    mr.se3ToVec(mr.MatrixLog6(np.dot(mr.TransInv(Tsb), T))))
        err = np.linalg.norm([Vs[0], Vs[1], Vs[2]]) > eomg \
              or np.linalg.norm([Vs[3], Vs[4], Vs[5]]) > ev
    return (thetalist, not err,i)

In [30]:
from ik.kinematics.fk import FK,FK_batch_full,_Slist_np,_M_HOME_np

i=1
IKinSpace_with_iters(_Slist_np,_M_HOME_np,FK(test_ds.q_target[i]),q_predict[i],0.001,0.001)

(array([-1.44960048, -1.1817319 ,  0.04990294,  1.92684414,  0.15471801,
         1.23075421]),
 True,
 1)

In [35]:
import numpy as np
! pip install tdqm
from tqdm import tqdm
iterations_list = []
convergence_list = []

for i in tqdm(range(len(q_predict))): # Iterate through all predicted configurations
    # Use the true target pose for the IKinSpace_with_iters function
    target_T = FK(test_ds.q_target[i])
    initial_q = q_predict[i]

    # Call IKinSpace_with_iters
    thetalist_out, converged, num_iters = IKinSpace_with_iters(_Slist_np, _M_HOME_np, target_T, initial_q, 0.001, 0.001)

    iterations_list.append(num_iters)
    convergence_list.append(converged)


average_iterations = np.mean(iterations_list)
percentage_convergence = np.sum(convergence_list) / len(convergence_list) * 100

print(f"Average iterations: {average_iterations:.2f}")
print(f"Percentage of convergence: {percentage_convergence:.2f}%")

100%|██████████| 30000/30000 [01:46<00:00, 281.67it/s]

Average iterations: 1.50
Percentage of convergence: 100.00%


In [36]:
iterations_list

[2,
 1,
 1,
 1,
 2,
 1,
 2,
 1,
 1,
 1,
 2,
 2,
 1,
 1,
 1,
 1,
 2,
 1,
 2,
 2,
 2,
 2,
 2,
 1,
 2,
 1,
 2,
 3,
 2,
 1,
 2,
 1,
 3,
 1,
 2,
 1,
 1,
 2,
 1,
 1,
 1,
 1,
 2,
 1,
 1,
 1,
 2,
 1,
 2,
 1,
 2,
 1,
 1,
 1,
 2,
 2,
 2,
 1,
 2,
 1,
 2,
 2,
 2,
 1,
 2,
 2,
 1,
 1,
 2,
 1,
 1,
 2,
 2,
 1,
 1,
 1,
 2,
 1,
 2,
 1,
 2,
 1,
 2,
 2,
 1,
 1,
 1,
 1,
 4,
 2,
 1,
 2,
 2,
 1,
 1,
 2,
 1,
 1,
 3,
 2,
 2,
 1,
 2,
 2,
 1,
 2,
 2,
 1,
 4,
 1,
 2,
 1,
 1,
 2,
 1,
 2,
 1,
 1,
 2,
 1,
 1,
 1,
 2,
 2,
 1,
 2,
 1,
 1,
 1,
 1,
 1,
 2,
 1,
 2,
 2,
 1,
 1,
 1,
 2,
 2,
 1,
 2,
 1,
 2,
 1,
 2,
 2,
 1,
 1,
 1,
 2,
 2,
 1,
 1,
 2,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 2,
 3,
 1,
 1,
 1,
 1,
 1,
 2,
 1,
 1,
 1,
 2,
 1,
 2,
 2,
 1,
 1,
 2,
 1,
 1,
 2,
 1,
 1,
 2,
 2,
 1,
 1,
 1,
 2,
 1,
 1,
 2,
 2,
 1,
 1,
 2,
 2,
 2,
 1,
 1,
 1,
 2,
 1,
 2,
 2,
 2,
 1,
 1,
 1,
 2,
 1,
 1,
 1,
 1,
 1,
 2,
 2,
 1,
 2,
 1,
 1,
 1,
 2,
 1,
 2,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 2,
 2,
 2,
 5,
 2,
 2,
 1,
 1,
 2,


In [28]:
FK(test_ds.q_target[i])
q_predict[i]

array([-1.487952  , -1.1646061 ,  0.04176199,  2.0768769 ,  0.15130937,
        1.1554229 ], dtype=float32)